In [ ]:
cd /home/조기정/project/RAG_LLM/
# conda activate RAG_LLM
conda activate Qwen2.5










In [ ]:
"""
@@ PDF paser도 좋은거 골라야 함

pdf_text_extractor.py
────────────────────────────────────────────────
• data/            ⟶ PDF 원본이 들어있는 루트 폴더
• extracted_texts/ ⟶ PDF별 추출 결과(.txt) 저장 위치
────────────────────────────────────────────────
pip install pymupdf tqdm fitz


python pdf_text_extractor.py  

"""

In [ ]:
# cd /home/조기정/project/RAG_LLM/src/test

from pathlib import Path
import fitz           # PyMuPDF
import json
from tqdm import tqdm  # 진행률 표시용 (선택)

ROOT_PDF_DIR   = Path("rowdata")
OUTPUT_TXT_DIR = Path("extracted_texts")
META_JSON_PATH = Path("extracted_texts/_extraction_meta.json")  # 진행 내역‧검증용

OUTPUT_TXT_DIR.mkdir(parents=True, exist_ok=True)

def extract_text_from_pdf(pdf_path: Path) -> str:
    """한 PDF(멀티 페이지)의 텍스트를 전부 이어붙여 반환"""
    doc  = fitz.open(pdf_path)
    text = []
    for page in doc:                        # 페이지 순회
        page_text = page.get_text("text")   # layout 無, 순수 텍스트
        text.append(page_text.strip())
    return "\n\n".join(text)

def main():
    # 이전에 완료한 PDF는 건너뛰기 위해 메타 파일 로드
    done_files = {}
    if META_JSON_PATH.exists():
        done_files = json.loads(META_JSON_PATH.read_text(encoding="utf-8"))

    new_meta = {}

    pdf_paths = list(ROOT_PDF_DIR.rglob("*.pdf"))
    if not pdf_paths:
        print("처리할 PDF가 없습니다.")
        return

    for pdf_path in tqdm(pdf_paths, desc="PDF 전처리"):
        pdf_rel  = pdf_path.relative_to(ROOT_PDF_DIR)             # data 하위 상대경로
        txt_path = OUTPUT_TXT_DIR / pdf_rel.with_suffix(".txt")   # .txt 경로 매핑

        # 이미 추출 완료된 파일이면 건너뜀
        if str(pdf_rel) in done_files and txt_path.exists():
            new_meta[str(pdf_rel)] = done_files[str(pdf_rel)]
            continue

        # 추출
        try:
            pdf_text = extract_text_from_pdf(pdf_path)
            txt_path.parent.mkdir(parents=True, exist_ok=True)
            txt_path.write_text(pdf_text, encoding="utf-8")

            # 간단 검증: 글자 수/줄 수 저장
            lines = pdf_text.splitlines()
            info = {
                "chars": len(pdf_text),
                "lines": len(lines),
                "preview": pdf_text[:200].replace("\n", " ") + "…"  # 앞 200자
            }
            new_meta[str(pdf_rel)] = info

            # 콘솔에도 일부 확인
            print(f"\n✅ [{pdf_rel}] → {info['chars']} chars, {info['lines']} lines")
            print(f"   preview: {info['preview']}\n")

        except Exception as e:
            print(f"❌ [{pdf_rel}] 추출 실패: {e}")

    # 메타정보 저장(누적)
    META_JSON_PATH.write_text(json.dumps(new_meta, ensure_ascii=False, indent=2),
                              encoding="utf-8")
    print("\n📝 전처리 완료 – 결과는 extracted_texts/ 폴더와 _extraction_meta.json을 확인하세요.")

In [2]:

if __name__ == "__main__":
    main()


NameError: name 'main' is not defined

In [ ]:

"""
리눅스 docker 설치 
sudo wget -qO- http://get.docker.com/
sudo apt-get update
sudo apt-get install docker.io
sudo ln -sf /usr/bin/docker.io /usr/local/bin/docker
"""

"""
docker에 milvus 설치 

Download the installation script
curl -sfL https://raw.githubusercontent.com/milvus-io/milvus/master/scripts/standalone_embed.sh -o standalone_embed.sh

Start the Docker container
bash standalone_embed.sh start



 => http://127.0.0.1:9091/
"""


"""
FAISS 백터 DB 구축
!pip install faiss-cpu
! pip install sentence_transformers
! pip install --upgrade --force-reinstall sentence-transformers
! pip install pandas
! pip install pyarrow
! pip install dill
! pip install aiohttp
! pip install numpy
! pip install accelerate
"""

"""
milvus 백터 DB 구축
python3 -m pip install pymilvus==2.6.0b0
pip install "pymilvus[model]"

python /home/조기정/project/RAG_LLM/src/test/milvus_ingest_chunked.py
"""



  Using cached accelerate-1.8.1-py3-none-any.whl.metadata (19 kB)
Using cached accelerate-1.8.1-py3-none-any.whl (365 kB)


In [5]:
from pymilvus import MilvusClient

client = MilvusClient("milvus_demo.db")

In [ ]:
"""#!/usr/bin/env python3
# milvus_ingest_chunked.py

"""
PDF에서 추출된 텍스트(.txt)를 512토큰 이하 청크로 분할하여
임베딩한 후 Milvus 컬렉션에 저장하는 스크립트입니다.
"""

import json
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from pymilvus import (
    connections, FieldSchema, CollectionSchema, DataType,
    Collection, utility
)
from transformers import AutoTokenizer, AutoModel

# ─────────── 설정 ───────────
MILVUS_HOST = "localhost"
MILVUS_PORT = "19530"
COL_NAME    = "pdf_chunks"
MODEL_DIR   = "/home/조기정/project/RAG_LLM/src/test/embedding"
TEXT_DIR    = Path("extracted_texts")

MAX_TOKENS  = 512   # 청크당 최대 토큰 수
OVERLAP     = 64    # 청크 간 중복 토큰 수
# ──────────────────────────

# 0) Milvus 연결
connections.connect("default", host=MILVUS_HOST, port=MILVUS_PORT)

# 1) 토크나이저·모델 준비
device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer= AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
model    = AutoModel.from_pretrained(MODEL_DIR, trust_remote_code=True).to(device).eval()
emb_dim  = model.config.hidden_size

# 2) 컬렉션 스키마 정의 & 생성
if utility.has_collection(COL_NAME):
    utility.drop_collection(COL_NAME)

fields = [
    FieldSchema(name="pk",         dtype=DataType.INT64,  is_primary=True),
    FieldSchema(name="embedding",  dtype=DataType.FLOAT_VECTOR, dim=emb_dim),
    FieldSchema(name="path",       dtype=DataType.VARCHAR, max_length=500),
    FieldSchema(name="chunk_idx",  dtype=DataType.INT64),
]
schema = CollectionSchema(fields, description="PDF text chunks")
collection = Collection(COL_NAME, schema)
# 인덱스 생성 (HNSW)
collection.create_index(
    field_name="embedding",
    index_params={"metric_type":"IP","index_type":"HNSW","params":{"M":16,"efConstruction":200}}
)
collection.load()

# 3) mean pooling 정의
def mean_pooling(outputs, mask):
    token_emb = outputs.last_hidden_state
    mask_exp   = mask.unsqueeze(-1).expand(token_emb.size()).float()
    summed     = torch.sum(token_emb * mask_exp, dim=1)
    counts     = torch.clamp(mask_exp.sum(dim=1), min=1e-9)
    return summed / counts

# 4) 텍스트를 토큰 기반으로 청크 분할
def chunk_text(text:str, max_tokens=MAX_TOKENS, overlap=OVERLAP):
    ids = tokenizer.encode(text, add_special_tokens=False, return_tensors="pt")[0]
    total, chunks, start = ids.size(0), [], 0
    while start < total:
        end = min(start + max_tokens, total)
        chunks.append(tokenizer.decode(ids[start:end], skip_special_tokens=True))
        start += max_tokens - overlap
    return chunks

# 5) TXT 파일 순회 → 임베딩 → Milvus 삽입
pk = 0
for txt_path in TEXT_DIR.rglob("*.txt"):
    rel = str(txt_path.relative_to(TEXT_DIR))
    text= txt_path.read_text(encoding="utf-8")
    for cidx, chunk in enumerate(chunk_text(text)):
        # 임베딩
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True,
                           max_length=MAX_TOKENS, padding="longest").to(device)
        with torch.no_grad():
            out = model(**inputs)
        emb = mean_pooling(out, inputs["attention_mask"])
        emb = F.normalize(emb, p=2, dim=1).cpu().numpy().astype("float32")

        # Milvus에 삽입
        collection.insert([
            [pk],                # pk
            emb.tolist(),        # embedding
            [rel],               # path
            [cidx],              # chunk_idx
        ])
        pk += 1

print(f"✅ Milvus 컬렉션 '{COL_NAME}' 저장 완료. 총 삽입 문서 수: {pk}")
"""

/home/조기정/.conda/envs/Qwen/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MilvusException: <MilvusException: (code=2, message=Fail connecting to server on localhost:19530, illegal connection params or server unavailable)>

In [ ]:
"""
Milvus에서 사용자 쿼리를 임베딩하여 유사도 검색 후
해당 청크의 텍스트 스니펫을 반환하는 스크립트입니다.


conda activate Qwen2.5
python /home/조기정/project/RAG_LLM/src/test/milvus_search_chunked.py  "인사규정 제21조부터 제24조까지 요약해줘." --top_k 5
"""

In [ ]:
#!/usr/bin/env python3
# milvus_search_chunked.py


"""
conda install -c milvus-io milvus
"""

import json
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from pymilvus import (
    connections, FieldSchema, CollectionSchema, DataType,
    Collection, utility
)
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm  # 진행률 표시용

# ─────────── 설정 ───────────
MILVUS_HOST = "localhost"
MILVUS_PORT = "19530"
COL_NAME    = "pdf_chunks"
MODEL_DIR   = "/home/조기정/project/RAG_LLM/src/test/embedding"
TEXT_DIR    = Path("extracted_texts")

MAX_TOKENS  = 512   # 청크당 최대 토큰 수
OVERLAP     = 64    # 청크 간 중복 토큰 수
# ──────────────────────────

# 0) Milvus 연결
connections.connect("default", host=MILVUS_HOST, port=MILVUS_PORT)

# 1) 토크나이저·모델 준비
device   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer= AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
model    = AutoModel.from_pretrained(MODEL_DIR, trust_remote_code=True).to(device).eval()
emb_dim  = model.config.hidden_size

# 2) 컬렉션 스키마 정의 & 생성
if utility.has_collection(COL_NAME):
    utility.drop_collection(COL_NAME)

fields = [
    FieldSchema(name="pk",         dtype=DataType.INT64,  is_primary=True),
    FieldSchema(name="embedding",  dtype=DataType.FLOAT_VECTOR, dim=emb_dim),
    FieldSchema(name="path",       dtype=DataType.VARCHAR, max_length=500),
    FieldSchema(name="chunk_idx",  dtype=DataType.INT64),
]
schema = CollectionSchema(fields, description="PDF text chunks")
collection = Collection(COL_NAME, schema)
# 인덱스 생성 (HNSW)
collection.create_index(
    field_name="embedding",
    index_params={"metric_type":"IP","index_type":"HNSW","params":{"M":16,"efConstruction":200}}
)
collection.load()

# 3) mean pooling 정의
def mean_pooling(outputs, mask):
    token_emb = outputs.last_hidden_state
    mask_exp   = mask.unsqueeze(-1).expand(token_emb.size()).float()
    summed     = torch.sum(token_emb * mask_exp, dim=1)
    counts     = torch.clamp(mask_exp.sum(dim=1), min=1e-9)
    return summed / counts

# 4) 텍스트를 토큰 기반으로 청크 분할
def chunk_text(text:str, max_tokens=MAX_TOKENS, overlap=OVERLAP):
    ids = tokenizer.encode(text, add_special_tokens=False, return_tensors="pt")[0]
    total, chunks, start = ids.size(0), [], 0
    while start < total:
        end = min(start + max_tokens, total)
        chunks.append(tokenizer.decode(ids[start:end], skip_special_tokens=True))
        start += max_tokens - overlap
    return chunks

# 5) TXT 파일 순회 → 임베딩 → Milvus 삽입
pk = 0
# 텍스트 파일 리스트 수집
text_files = list(TEXT_DIR.rglob("*.txt"))
total_files = len(text_files)

for idx, txt_path in enumerate(tqdm(text_files, desc="TXT 파일 처리"), 1):
    rel = str(txt_path.relative_to(TEXT_DIR))
    print(f"[{idx}/{total_files}] {rel} 처리 중...")
    text= txt_path.read_text(encoding="utf-8")
    for cidx, chunk in enumerate(chunk_text(text)):
        # 임베딩
        inputs = tokenizer(chunk, return_tensors="pt", truncation=True,
                           max_length=MAX_TOKENS, padding="longest").to(device)
        with torch.no_grad():
            out = model(**inputs)
        emb = mean_pooling(out, inputs["attention_mask"])
        emb = F.normalize(emb, p=2, dim=1).cpu().numpy().astype("float32")

        # Milvus에 삽입
        collection.insert([
            [pk],                # pk
            emb.tolist(),        # embedding
            [rel],               # path
            [cidx],              # chunk_idx
        ])
        pk += 1

print(f"✅ Milvus 컬렉션 '{COL_NAME}' 저장 완료. 총 삽입 문서 수: {pk}")


/home/조기정/.conda/envs/Qwen/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  2.02it/s]
usage: ipykernel_launcher.py [-h] [--top_k TOP_K] query
ipykernel_launcher.py: error: the following arguments are required: query


SystemExit: 2

/home/조기정/.conda/envs/Qwen/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3675: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# FAISS 결과

"USER : “인사위원회 회의록 초안을 작성해줘.”
RAG :
다음은 서울시설공단 인사규정 중 제6조(인사위원회 및 추천심사위원회<2005.6.30>) 전문입니다.
“① 인사관리의 합리화를 기하기 위하여 인사위원회를 둔다.
② 직원승진과 내부직위공모 등을 위하여 필요한 경우, 선발의 타당성과 공정성 확보를 위해 공적 등을 심사하는 추천심사위원회를 둔다.(2005.6.30 신설)
③ 인사위원회 및 추천심사위원회의 운영에 관하여 필요한 사항은 내규로 정한다.(2005.6.30)”

위 내용을 바탕으로, USER의 요청에 따라서 인사위원회 회의록 초안을 작성해주세요."	"USER : “채용방법 안내 문서를 만들어줘.”
RAG :
다음은 서울시설공단 인사규정 중 제8조(채용방법) 전문입니다.
“① 직원의 신규채용은 공개경쟁시험에 의한다. 다만, 다음 각호의 1에 해당하는 경우에는 경력경쟁시험에 의해 채용할 수 있다.

공개경쟁시험에 의하여 임용하는 것이 부적당한 경우에 임용예정 직무수행에 필요한 당해 분야의 국가가 인정하는 자격증 소지자를 임용하는 경우

공개경쟁시험에 의하여 결원보충이 곤란한 특수직렬의 직원을 임용하는 경우

(삭제 2003.11.7)

다른 법령에 정하여진 바에 따라 임용하는 경우

정원의 개폐 또는 예산의 감소 등에 의하여 폐직 또는 감원으로 퇴직하거나 신체·정신상의 장애로 인하여 휴직기간 만료로 퇴직한 날로부터 2년 이내에 퇴직 당시 직급 또는 그 하위 직급에 재임용하는 경우

긴급충원이 불가피한 경우

(삭제 2013.12.9)

국가 및 지방자치단체의 공무원과 그 투자기관에 재직하는 자를 채용하는 경우 (개정 ’03.11.7, 2011.4.1)”

위 내용을 바탕으로, USER의 요청에 따라서 채용방법 안내 문서를 만들어주세요.
"	"USER : “인사규정 제21조부터 제24조까지 요약해줘.”
RAG :
다음은 서울시설공단 인사규정 중 제21조~제24조 전문입니다.
[제21조]
“① 승진은 동일직렬의 차하위 직급에서 인사위원회, 추천심사위원회 등을 통한 심사승진함을 원칙으로 실시하되 1회에 1직급, 승급은 1년에 1호봉을 원칙으로 한다. 다만 승진은 3월 중 실시하되 필요할 경우에는 그 시기를 조정하여 시행할 수 있다.(2005.6.30, 2010.6.1)USER : “인사규정 제21조부터 제24조까지 요약해줘.”
RAG :
다음은 서울시설공단 인사규정 중 제21조~제24조 전문입니다.
[제21조]
“① 승진은 동일직렬의 차하위 직급에서 인사위원회, 추천심사위원회 등을 통한 심사승진함을 원칙으로 실시하되 1회에 1직급, 승급은 1년에 1호봉을 원칙으로 한다. 다만 승진은 3월 중 실시하되 필요할 경우에는 그 시기를 조정하여 시행할 수 있다.(2005.6.30, 2010.6.1)
② 제1항의 규정에 의한 승진의 방법 및 기준은 내규로 정한다.(2004.8.26, 2005.6.30, 2010.6.1)
③ 8급 직원이 입사 후 승진최소소요연수의 3배수(6년)이 경과하여도 승진하지 못한 경우에는 인사위원회의 심의를 거쳐 1회에 한하여 1직급(7급(보))을 근속승진시키며, 시기는 매분기 초로 한다.<신설 2019.4.19.>”

[제22조]
“① 직원을 승진시키고자 하는 경우에는 당해 직급의 승진후보자 명부의 서열을 참작하여 승진하고자 하는 결원 범위 내에서 실시한다.
② 승진후보순위의 결정은 근무성적평정, 경력평정, 교육훈련평정, 가감평정 등에 의한다. 단, 경영혁신에 현저한 공적이 있으며 그 공적이 심사위원회 심의에서 인정된 직원에 대하여 이사장은 그 직원이 해당 직급에 재임하는 동안에 승진후보순위 결정 시, 서열명부상 총점의 3% 이내의 점수를 별도로 정하여 가산할 수 있다.(2004.8.26, ’06.1.9 단서 신설)
③ 제2항의 규정에 의한 절차, 방법 및 세부사항은 내규로 정한다.”

[제23조]
“① 직원이 승진함에 있어서는 다음 각호의 기간 이상을 당해 직급에 근무하여야 한다.('99.7.2, 2010.6.1)

2급 내지 4급: 3년 이상

5급 내지 6급: 2년 6개월 이상(개정 2014.4.1)

7급 내지 8급: 2년 이상(신설 2014.4.1)<개정 2019.4.19.>

삭제<2019.4.19.>
② 삭제<'99.7.2>
③ 제1항의 직급별 승진 소요년수 산정은 휴직기간, 직위해제기간, 정직기간, 강등에 따라 직무에 종사하지 아니한 기간을 포함하지 아니한다. 다만, 제33조 제3호·제6호·제9호의2의 휴직기간은 예외로 하고, 제7호의 경우에는 휴직기간 최초 1년만을, 제10호의 경우에는 휴직기간의 50%를 근무년수에 산입하며, 징계처분 또는 직위해제처분을 받은 자가 처분 사유가 된 사건에 대해 법원 판결 또는 노동위원회 결정으로 무죄·무효·취소된 경우 그 기간을 근무년수에 산입한다.”

[제24조]
“① 징계등 처분요구, 징계등 의결요구, 징계등 처분, 직위해제 및 휴직기간 중에 있는 직원은 승진 및 승급할 수 없다. 단, 업무상 부상 또는 질병으로 판정받아 휴직 중인 자나 육아휴직자의 승급은 예외로 한다.(’03.11.7 본항개정, 2018.5.21.)”

위 내용을 바탕으로, USER의 요청에 따라서 인사규정 제21조~제24조를 요약해주세요.
② 제1항의 규정에 의한 승진의 방법 및 기준은 내규로 정한다.(2004.8.26, 2005.6.30, 2010.6.1)
③ 8급 직원이 입사 후 승진최소소요연수의 3배수(6년)이 경과하여도 승진하지 못한 경우에는 인사위원회의 심의를 거쳐 1회에 한하여 1직급(7급(보))을 근속승진시키며, 시기는 매분기 초로 한다.<신설 2019.4.19.>”
"	"USER : “인사규정 제33조(휴직사유와 기간)만 요약해줘.”
RAG :
다음은 서울시설공단 인사규정 중 제33조(휴직사유와 기간) 전문입니다.
“① 직원이 다음 각호의 1에 해당할 때에는 휴직을 명할 수 있다.

신체·정신상의 장애로 장기요양을 요할 때(불임·난임 치료 포함): 1년 이내, 부득이한 경우 1년 연장 가능(2005.6.30, 2015.3.31.)

병역법 등 법령에 의해 징집 또는 소집된 경우: 징집·소집·동원 기간(2005.6.30)

업무로 인한 질병 또는 공상으로 1개월 이상 장기요양을 요할 때: 3년 이내(2005.6.30, 2015.3.31.)

천재지변·전시사변 등으로 생사 또는 소재 불명 시: 6개월 이내(2005.6.30)

삭제(2005.6.30)

직무 관련 분야 해외유학: 2년 이내(2005.6.30, 2015.3.31.)

임신 중 모성 보호 또는 만 8세 이하·초등2학년 이하 자녀 양육 위해 육아휴직: 3년(횟수 구분 없이 분할 사용 가능)

공단 지정 연구·교육기관 연수: 3년 이내(2005.6.30 신설)

부모·배우자·자녀·배우자 부모 간호 시: 1년 이내(본호신설 ’07.4.11)
9의2. 남녀고용평등법 제22조의2 가족돌봄휴직: 연간 90일 이내(분할 사용 시 30일 이상 단위, 무급휴가 포함)

개인적 필요의 해외유학: 3년 이내, 부득이한 경우 2년 이내 연장 가능<신설 ’09.3.1, 개정 2016.2.12>”

위 내용을 바탕으로, USER의 요청에 따라서 제33조만 요약해주세요.
"	"USER : “면직 사유에는 어떤 것들이 있나요?”
RAG :
다음은 서울시설공단 인사규정 중 제37조(직위해제) 및 제12조(결격사유) 전문입니다.

[제37조]
“① 이사장은 직원이 다음 각호의 1에 해당할 때에는 직위를 해제할 수 있다.

형사사건으로 공소가 제기된 자. 단, 약식명령이 청구된 자는 제외한다.(2005.6.30)

직무수행능력이 현저히 부족하거나 근무성적이 극히 불량한 자

소속 직원에 대한 감독능력이 부족한 자

징계등 의결요구 중인 자(개정 2018.5.21.)

공단 직원으로서 품위를 훼손하거나 현저히 공단 이익에 반한 행위를 한 자.”

[제12조]
“① 다음 각호의 1에 해당할 때에는 직원으로 채용할 수 없다.

피성년후견인 또는 피한정후견인(2015.3.31)

파산선고를 받고 복권되지 아니한 자(2015.3.31)

금고 이상의 형을 받고 그 집행이 종료되거나 집행받지 아니하기로 확정된 후 5년을 경과하지 아니한 자(2015.3.31)

금고 이상의 형을 받고 집행유예기간이 종료된 날로부터 2년을 경과하지 아니한 자(2015.3.31)

금고 이상의 형의 선고유예를 받은 자

「부패방지 및 국민권익위원회 설치 및 운영에 관한 법률」제82조에 따른 비위면직자 등(2013.12.9, 2015.3.31, 2017.7.25.)

법원 판결 또는 다른 법률에 따라 자격이 상실·정지된 자(2015.3.31)
7의2. 성폭력범죄 처벌 등에 관한 특례법 제2조에 따른 죄(100만원 이상 벌금형 확정 후 3년 미경과)

병역의무자로서 병역 기피 사실이 있는 자

정보통신망 이용촉진 및 정보보호 등에 관한 법률 제74조 제1항 제2·3호 위반 시 100만원 이상 벌금형 확정 후 3년 미경과.”

위 내용을 바탕으로, USER의 질문에 답변해주세요."	"USER : “수습임용 기간 중 해고 사유는 무엇인가요?”
RAG :
다음은 서울시설공단 인사규정 중 제11조(수습임용) 전문입니다.
“① 4급 이하 직원을 신규채용할 때는 3개월 미만의 수습기간을 둘 수 있다.(’03.11.7 단서삭제)<개정 2019.6.27>
② 신규채용된 자가 수습기간 중 다음 각호의 1에 해당할 때에는 채용하지 아니한다.

근무성적이 불량할 때

공단의 제규정을 위반하였을 때

제12조의 결격사유에 해당할 때.”

위 내용을 바탕으로, USER의 질문에 답변해주세요.
"


In [ ]:
# Milvus 결과
ID 401 | score=0.8394 | path=인사규정시행내규_20250124.txt | chunk=63
snippet:
 내규는 공포한 날부터 시행한다.
부    칙(2018. 9. 4.)
이 내규는 공포한 날부터 시행한다.
부    칙(2018.12.28.)
이 내규는 공포한 날부터 시행한다.
부      칙(2019.4.16.)(직제규정 제956호)
제1조(시행일) 이 규정은 공포한 날부터 시행한다. 다만, 제10조 제2항의 개정규정은 2019년 1월 1일부터 적용
하고, 제10조 제1항 및 제3항, 제14조, 별표2의 개정규정은 2019년 4월 1일부터 적용한다. 
제2조(다른 규정의 개정) ① 생략
② 인사규정시행내규 일부를 다음과 같이 개정한다. 
   별지 제8호 서식 중 “인사처장”을 “인사노무처장”으로 한다.
③부터 ⑥까지 생략
부    칙(2019.4.19.)
제1조(시행일) 이 내규는 공포한 날부터 시행하되, 2019년 1월 1일부터 적용한다.
제2조(징계에 관한 적용례) 제49조의2, 제53조 및 제53조의3의 개정내규는 이 내규 공포일 이후 발생한 사유로 
징계 등을 받는 사람부터 적용한다.
제3조(승진후보자명부 작성에 대한 경과조치) 인사규정 개정에 따라 9급에서 8급, 8급에서 7급(보)로 직급전환(승
진)된 직원의 승진후보자명부 작성 시 기준이 되는 현직급승진일자는 직급전환 이전 직급의 승진일자로 적용한
다.
부    칙(2019.9.30.)
제1조(시행일) 이 내규는 공포한 날부터 시행한다
────────────────────────────────────────────────────────────
ID 516 | score=0.8391 | path=직제규정_20240828.txt | chunk=28
snippet:
. 
부    칙(2019. 7. 1.)
이 규정은 공포한 날부터 시행한다.
부    칙(2019. 9.20.)

이 규정은 공포한 날부터 시행한다.
부    칙(2019.12.16.)
이 규정은 공포한 날부터 시행한다.
부    칙(2020.4.2.)
이 규정은 공포한 날부터 시행한다.
부    칙(2021.4.1.)
이 규정은 공포한 날부터 시행한다.
부    칙(2021.7.1.)
이 규정은 공포한 날부터 시행한다.
부    칙(2021.12.31.)
제1조(시행일) 이 규정은 공포한 날부터 시행한다.
제2조(다른 규정의 개정) ① 인사규정 일부를 다음과 같이 개정한다. 
제4조제2항, 제4조의2제1항, 별표1-1 중 “조무”를 “실무”로 한다.
② 보수규정시행내규 일부를 다음과 같이 개정한다.
제7조제2호 나목, 같은 조 제3호 나목 중 “조무”를 “실무”로 한다.
부    칙(2022.4.13.)
이 규정은 공포한 날부터 시행한다.
부    칙(2023.4.10.)
제1조(시행일) 이 규정은 공포한 날부터 시행한다.
제2조(다른 규정의 개정) ① 직제규정시행내규를 다음과 같이 개정한다. 
제11조 제명을 “인재원”에서 “인재문화원”으로 하고, 같은 조 제1항 및 제2항 중 “인재원장”을 각각 “인재문
화원장”으로 한다.
② 사무위임전결규정시행내규를 다음과 같이 개정한다.
별표1 중 “Ⅶ.인재원”을 “Ⅶ.인재문화원”으로 하고, Ⅶ. 제5호 제목 중 “인
────────────────────────────────────────────────────────────
ID 391 | score=0.8326 | path=인사규정시행내규_20250124.txt | chunk=53
snippet:
 제50조의 규정을 준용한다.
(개정 2015.5.7)
  ③ 인사담당 부서의 장은「부패방지 및 국민권익위원회의 설치와 운영에 관한 법률」제82조 제1항에 따른 비위면직자 
등에게 같은 조 제2항에 따라 취업이 제한된다는 사실을 지체 없이 안내하여야 한다. <신설 2018.9.4.>
제56조(징계등의 집행) 이사장은 징계등 의결서를 받은 날로부터 15일 이내에 이를 집행하여야 한다. <개정 
2018.5.21.>
[제목개정 2018.5.21.]
제57조(상벌자명부 작성비치 및 관리) 이 내규 및 다른 규정 또는 법령에 의하여 포상 또는 처벌을 받은 자에 
대하여는 인사담당부서에서 그 명부를 작성 비치하여 관리한다.
제10장  보 칙
제58조(정년기준일) ①규정 제29조의 규정에 의한 정년의 기준일은 주민등록상 만60세가 되는 해의 12월 31일
로 한다.(개정 2008.2.18, 2011.4.1, 2016.4.5)
  ② 제1항의 규정에 의한 정년자에게는 예산의 범위 내에서 공로표창장 또는 공로패를 수여한다.
제58조의2(명예퇴직) ①규정 제30조의2의 규정에 의하여 명예퇴직하고자 하는 경우에는 매월 10일까지 소속 부서장이 
해당직원의 명예퇴직신청서(별지 제13호 서식)를 인사부서의 장에게 제출하여야 한다.(2004. 8.26)
  ② 명예퇴직을 신청한 자는 임용권자의 승인시까지 근무하여야 한다.
  ③
────────────────────────────────────────────────────────────
ID 241 | score=0.8306 | path=인사규정_20250604.txt | chunk=36
snippet:
50조의 기간이 경과하거나 그 잔여기
간이 1개월 미만인 경우에는 제50조의 기간은 그 조사나 수사가 종료한 날로부터 1개월 경과한 날에 만료되
는 것으로 본다. <개정 2018.5.21.>
제52조(상벌위원회) 직원의 포상 및 징계등에 관한 사항을 공정하게 처리하기 위하여 상벌위원회를 둔다. <개정 
2018.5.21.>
제8장  복무
제53조(복무) 복무에 관한 사항은 취업규칙에 의한다.

제9장  보수
제54조(보수) 보수에 관한 사항은 보수규정에 의한다.
제10장  보칙
제55조(신원보증) ① 직원은 재직 중 금전상 또는 재정상의 사고를 보증하기 위하여 신원보증 보험에 가입하여야 
한다.
  ② 신원보증에 대한 세부사항은 내규로 정한다.
제56조(인사기록) ① 인사담당부서는 직원의 직무와 신상의 제반사항을 기록한 인사기록을 작성 비치하거나 전산
화된 자료로 유지관리하여야 한다.(2001.4.23)
  ② 직원의 직무 및 신상 등 제반 변동사항 발생시 해당부서장은 지체없이 인사담당 부서의 장에게 통보하여야 
한다.('99.7.2)  
제57조(위임규정) 이 규정 시행에 관하여 필요한 사항은 내규로 정하며, 조사직, 서비스직 직원에 관한 사항은 
별도 규정으로 정할 수 있다. (2000.7.25, 2008.2.18, 2010.6.1, 2015.3.31)
부    칙
①(시행일) 이 규정은 1983년 12월 26일
────────────────────────────────────────────────────────────
ID 341 | score=0.8305 | path=인사규정시행내규_20250124.txt | chunk=3
snippet:

2025. 1.24 내규 제506호
제1장  총칙
제1조(목적) 이 내규는 인사규정(이하규정이라 한다)의 시행에 관하여 필요한 사항을 규정함을 목적으로 한다.
제2조(임용권의 위임) 규정 제3조 제2항의 규정에 의하여 임용권을 위임받은 본부장 및 처(원․실․장)장이 그 인사
를 행한 경우에는 이를 지체 없이 이사장에게 보고하여야 한다.(2011.7.1)
제2장  채용
제3조(채용시험의 원칙) 규정 제8조에서 정한 경력경쟁시험에 의한 채용을 제외하고는 직급별, 직종별로 시험을 
실시함을 원칙으로 한다. 다만, 직무에 따라 동일직종 내 채용분야를 세분화하여 채용할 수 있다. ('99.7.2, 
2011.4.1.) <개정 2022.3.15.>
제3조의2(채용계획의 사전 통보) 직원을 채용하고자 할 경우에는 채용의 필요성, 예상결원 및 정‧현원 현황, 채용
인원, 응시자격 요건, 필기시험 여부, 서류전형 심사 기준, 면접방법, 시험단계별 시험위원 위촉 계획 등이 포함
된 채용계획(인력수요의 변화 등으로 채용계획이 변경된 경우도 포함)을 공고예정일 15일전까지(필요시 서울시와 
협의하여 단축 가능) 서울시에 사전 통보 후 협의하여야 하며, 서울시가 제시한 의견을 원칙적으로 반영하여야 
한다. <개정 2019.4.19, 2019.9.30.>
[본조신설 2018.12.28.]

제4조(모집
────────────────────────────────────────────────────────────

In [ ]:
python /home/조기정/project/RAG_LLM/src/test/milvus_search_chunked.py  "인사위원회 회의록 초안을 작성해줘." --top_k 5

In [ ]:
ID 376 | score=0.8486 | path=인사규정시행내규_20250124.txt | chunk=38
snippet:
의 찬성으로 의결한다.
  ③ 위원장은 표결권을 가지며 가부동수인 경우에는 결정권을 가진다.

④ 위원회는 다음 각호의 1에 해당하는 경우에는 서면으로 의결할 수 있다. (2002.6.21 본항신설)
  1. 직원에게 불이익을 주지 아니하는 경미한 사항
  2. 긴급을 요하거나 위원회의 소집이 불가능하다고 판단되는 경우
제43조(간사 및 서기) ① 인사위원회의 간사와 서기는 인사담당 부서의 장이 지정하는 자로 한다.('99.7.2)(2005.11.1)
  ② 간사는 위원장의 명을 받아 인사위원회의 서무를 담당하고 회의록을 작성 보관하며 서기는 간사를 보좌한다.
제44조(감사의 출석) 인사위원회는 필요시 감사를 출석토록 요구할 수 있으며, 감사는 감사담당부서를 통해 대신 
의견을 진술할 수 있다. (2011.7.1)
제45조(보고 및 재심의 등) ① 인사위원회의 의결사항은 의결서를 첨부하여 지체없이 이사장에게 보고한다.
  ② 이사장은 제1항의 의결사항에 대하여 이의가 있을 때에는 보고를 받은 후 15일 이내에 재심의에 부의할 수 
있다.(개정 2015.5.7)
제45조의2(추천심사위원회의 구성 및 임무 등) ① 추천심사위원은 총14명 이상으로 하되 복수의 독립된 위원회
로 분리하여 운영할 수 있으며 위원장은 호선한다.(2005.6.30 신설)
  ② 추천심사위원회는 제23조 제2항에 의한 승진과 직위공모 및 고속승진 후보자의 공적 등을 심사하며 필요시 
제38조의 포상 및 가
────────────────────────────────────────────────────────────
ID 582 | score=0.8127 | path=재산관리규정_20240625.txt | chunk=15
snippet:
고 판단되는 경우에는 그러
하지 아니하다. 
 ② 위원장은 다음 각 호에 따라 위원회를 운영하여야 한다.
  1. 위원회는 특정한 위원에 의하여 부당하게 심의․의결되지 않도록 공정하게 운영하여야 한다. 
  2. 위원회는 법령, 규정 등에 규정된 심의․의결 등의 기한을 준수하여야 하며, 심의․의결 등의 기한이 없는 경
우에도 의사결정이 지연되지 않도록 한다.
 ③ 위원회의 세부운영에 관하여는 위원장의 별도 방침으로 정한다. 
제35조(회의록 작성) ① 위원회는 회의를 개최한 때에는 회의록(별지 제5호 서식)을 작성하여야 하며, 다음 각 
호의 사항을 기록하고 참석위원 전원의 서명을 받아 보관하여야 한다.
  1. 일시 및 장소
2. 참석자 및 배석자 명단
3. 진행 순서
4. 상정 안건
5. 발언 내용

6. 결정 사항 및 표결 내용
7. 그 밖에 위원회가 정하는 사항
 ② 위원회는 회의록을 확정한 날로부터 14일 이내에 해당 회의록을 공단 홈페이지에 공개하여야 한다. 다만, 
다음 각 호 어느 하나에 해당하는 경우에는 공개하지 아니할 수 있다.
   1. 다른 법령에 따라 비밀로 분류되거나 공개가 제한된 내용이 포함되어 있는 경우
 2. 공단의 보안상 비밀이 누설될 우려가 있다고 인정되는 경우
 3. 업무의 공정한 수행에 현저한 지장을 초래하는 경우
제36조(수당 등) 위원회에 참석한 외부위원에게는 공단 위원회 관리 규정 등 관련 규정에 따라 수당 등을 지급
할 수 있다.
제6장  보칙
제37조(대장 및 도면의 정비) 재산관리자는 재산의 변동 발생시 재산대장, 도면 및 이에 관련되는 증빙서류를 정
리하고 지체 없이 재산
────────────────────────────────────────────────────────────
ID 375 | score=0.8125 | path=인사규정시행내규_20250124.txt | chunk=37
snippet:
� 다음 각 호의 어느 하나에 해당되는 사람은 외부위원으로 위촉될 수 없다.
  1. 지방공무원법 제31조(결격사유)에 해당하는 사람
  2. 정당법에 따른 정당의 당원
  3. 지방의회 의원 [본항신설 2013.12.9] [전문개정 2011.1.11]
제41조(인사위원회의 임무<2005.6.30>) 인사위원회는 다음 각호의 사항을 심의한다.
  1. 인사제도와 인사에 관한 중요 기본방침
  2. 직원의 채용 및 승진에 관한 사항
  3. 3급 이상 및 4급 직위자의 징계등 재심의에 관한 사항(2004. 8.26)(2005.6.30)(2005.11.1)<개정 
2018.5.21.>
  4. 명예퇴직대상자 심의에 관한 사항(2004. 8.26)
  5. 상시·지속적 업무에 종사하는 기간제근로자의 정규직(무기계약직 등) 전환에 관한 사항 (본호신설 2016.12.30.)
  6. 기타 이사장이 심의 지시한 사항(2004. 8.26)[기존 제5호를 제6호로 이동, 2016.12.30.]
제42조(소집 및 의결) ①인사위원회는 위원장이 소집한다.
  ② 인사위원회는 그 구성원의 3분의 2이상 출석과 출석위원 과반수의 찬성으로 의결한다.
  ③ 위원장은 표결권을 가지며 가부동수인 경우에는 결정권을 가진다.

④ 위원회는 다음 각호의 1에 해당하는 경우에는 서면으로 의결할 수 있다. (2002
────────────────────────────────────────────────────────────
ID 59 | score=0.8122 | path=35._문서관리규정_20191128.txt | chunk=18
snippet:
최자, 회의일시, 장소, 참석자 및 인원수를 기입한다.
3) 세부내용란에서는 가)공단측의 설명(주장) 나)상대방 설명(주장) 다)결론 또는 합의사항 및 결정 
사항을 각 구분 기재하며 기타 문제점 등을 기록한다.
4) 기록보고서는 통상 1면으로 하나 부족할 경우에는 백지를 첨부 작성한다.
────────────────────────────────────────────────────────────
ID 581 | score=0.7929 | path=재산관리규정_20240625.txt | chunk=14
snippet:
 사항
  ② 제1항의 심의사항 중 다음 각 호의 어느 하나에 해당하는 경우에는 심의를 생략할 수 있다.
   1. 재산의 취득：1건당 예정가격 5천만원 이하
   2. 재산의 처분：1건당 예정가격 3천만원 이하
   3. 재산의 임대차 : 1건당 연 임대료 예정가격 3천만원 이하 신규 임대차
   4. 기타 관계 법령 또는 법원의 판결에 따른 재산의 변동 등
제33조(소집 및 의결) ① 위원장은 필요하다고 인정하는 경우 위원회를 소집한다.
 ② 위원회는 재적위원 3분의2 이상의 출석 및 출석위원 3분의2 이상의 찬성으로 의결한다. 
 ③ 회의에 출석한 위원은 회의 종료 후, 위원회 의결서(별지 제2호 서식), 직무윤리 준수서약서(별지 제3호 서
식)와 청렴서약서(별지 제4호 서식)에 서명 날인하여야 한다.
 ④ 위원은 대리인으로 하여금 회의에 출석하게 하거나 의결에 참여하게 하여서는 아니 된다. 다만 불가피한 사
유로 회의 참석이 불가능하여 사전에 위원장의 승인을 얻은 경우에는 대리인이 회의에 출석하여 의결할 수 있
다.
제34조(운영) ① 위원장은 회의 개최 7일 전까지 회의 일정과 안건 등을 위원에게 통보하여야 한다. 다만, 긴급
한 사유로 위원회를 개최하거나 회의 안건 공개 시 공정한 위원회 운영이 곤란하다고 판단되는 경우에는 그러
하지 아니하다. 
 ② 위원장은 다음 각 호에 따라 위원회를 운영하여야 한다.
  1. 위원회는 특정한 위원에 의하여 부당하게 심의․의결되지 않도록 공정하게 운영하여야 한다. 
  
────────────────────────────────────────────────────────────

In [ ]:
# conda create --name Qwen2.5 python=3.11.13
conda actiavate Qwen2.5
conda actiavate Qwen
cd /home/조기정/project/RAG_LLM



In [ ]:
""""""
hugging 모델 다운로드 기능 
"""

#!/usr/bin/env python3
# download_llm_models.py

import os
from transformers import AutoTokenizer, AutoModelForCausalLM

# 다운로드할 모델 리스트 (후보가 여러 개라면 여기에 추가)
MODEL_REPOS = [
    "Qwen/Qwen2.5-7B-Instruct-1M",
    # 예시: "Qwen/Qwen3-4B",
    #       "google/gemma-3n-E4B-it",
]

# 저장할 기본 경로
BASE_SAVE_DIR = "/home/조기정/project/RAG_LLM/src/test/llm_model"

def download_and_save_model(repo_id: str, save_dir: str):
    """
    repo_id Hugging Face 모델을 다운로드하여 save_dir에 저장한다.
    """
    print(f"▶ 다운로드 시작: {repo_id}")
    # 1) 토크나이저 로드 및 저장
    tokenizer = AutoTokenizer.from_pretrained(repo_id, trust_remote_code=True)
    tokenizer.save_pretrained(save_dir)
    # 2) 모델 로드 및 저장
    model = AutoModelForCausalLM.from_pretrained(repo_id, trust_remote_code=True)
    model.save_pretrained(save_dir)
    print(f"✔ 저장 완료: {save_dir}")

def main():
    os.makedirs(BASE_SAVE_DIR, exist_ok=True)

    for repo in MODEL_REPOS:
        # 모델명만 떼어내기 (슬래시 뒤 부분)
        model_name = repo.split("/")[-1]
        target_dir = os.path.join(BASE_SAVE_DIR, model_name)
        os.makedirs(target_dir, exist_ok=True)
        download_and_save_model(repo, target_dir)

if __name__ == "__main__":
    main()"""

▶ 스냅샷 다운로드 시작: Qwen/Qwen2.5-7B-Instruct-1M


Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

{"timestamp":"2025-07-15T02:33:45.354844Z","level":"WARN","fields":{"message":"Reqwest(reqwest::Error { kind: Request, url: \"https://transfer.xethub.hf.co/xorbs/default/0a0886bed087c3f21a39dbf31fdd8254b61ce1e7c3b87e09efe9e587fcbebb9f?X-Xet-Signed-Range=bytes%3D0-57757005&Expires=1752550394&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly90cmFuc2Zlci54ZXRodWIuaGYuY28veG9yYnMvZGVmYXVsdC8wYTA4ODZiZWQwODdjM2YyMWEzOWRiZjMxZmRkODI1NGI2MWNlMWU3YzNiODdlMDllZmU5ZTU4N2ZjYmViYjlmP1gtWGV0LVNpZ25lZC1SYW5nZT1ieXRlcyUzRDAtNTc3NTcwMDUiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkFXUzpFcG9jaFRpbWUiOjE3NTI1NTAzOTR9fX1dfQ__&Signature=Ojb9xBJs0XpQQhhYTEYMOnzdUdngGBPx6h0iFBLbMdPezqhGMrOEduh42JJeuvOK22AFXqfATeBVpZ-i6JR4h0I6HZ00tbzLIvZTKr424mk9U2HRt5v~1RaGuPIDE86bam6YmKZCbeOhcjHJwYprktgqjQYeWm5NnKtRf6SwrLs~HcZBHDjx-V6DEHfDwnt6vXVv1LPqbaFd42~t1BBc9OG~SSPdmbxmCrTiAW-l2wURiaQBYjfIJ6gGfRWggSvfALec3bBxqXXyTlYRyrciwgDfNrWpUFXZ4XgEMhB1D4IpsmQRE3uMQNe3imgx1CMgMsv9uycIWwqaKVYEEwcR5A__&Key-Pair-Id=K2L8F4GPSG1IFC\",

Fetching 15 files:  47%|████▋     | 7/15 [2:04:26<2:22:12, 1066.61s/it]


In [ ]:
google/gemma-3n-E4B-it
unsloth/gemma-2-9b-it


ValueError: Unrecognized model in /home/조기정/project/RAG_LLM/src/test/llm_model/Qwen2.5-7B-Instruct-1M. Should have a `model_type` key in its config.json, or contain one of the following strings in its name: albert, align, altclip, aria, aria_text, audio-spectrogram-transformer, autoformer, aya_vision, bamba, bark, bart, beit, bert, bert-generation, big_bird, bigbird_pegasus, biogpt, bit, bitnet, blenderbot, blenderbot-small, blip, blip-2, blip_2_qformer, bloom, bridgetower, bros, camembert, canine, chameleon, chinese_clip, chinese_clip_vision_model, clap, clip, clip_text_model, clip_vision_model, clipseg, clvp, code_llama, codegen, cohere, cohere2, colpali, conditional_detr, convbert, convnext, convnextv2, cpmant, csm, ctrl, cvt, d_fine, dab-detr, dac, data2vec-audio, data2vec-text, data2vec-vision, dbrx, deberta, deberta-v2, decision_transformer, deepseek_v3, deformable_detr, deit, depth_anything, depth_pro, deta, detr, diffllama, dinat, dinov2, dinov2_with_registers, distilbert, donu